# Sp3 RNA-seq — Machine-learning analysis

Predicting transcription-factor–target gene regulation (`log2FoldChange`) from
gene / TF / TF-family / tissue features, and quantifying how much the TF identity
contributes. Produces the manuscript **Supplementary Figure 2c**.

**How to run:** launch from the repository's `data/` directory so the input file
resolves (`final_filtered_gene_tf_data.csv`). All figures and tables are written to
`analysis_outputs/`.

Sections:
1. Exploratory data analysis
2. Exploratory classification of high-regulation genes
3. TF-family up/down regulation bias (binomial odds-ratio)
4. LMC vs MEF TF-family expression differences (Welch t-test)
5. Random-forest regression of log2FoldChange — **Supplementary Figure 2c**
6. Robustness: ablation & permutation importance of TF
7. Feature-set comparison
8. Leakage-safe target-encoded model
9. Generalization to unseen genes (gene-level holdout)
10. Mixed-effects model

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb

from scipy.stats import chi2_contingency, binomtest, norm, ttest_ind
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, r2_score,
                             mean_squared_error, mean_absolute_error)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Input file (run this notebook from the repository's data/ directory).
INPUT_CSV = "final_filtered_gene_tf_data.csv"
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_figure(base_name, dpi=300):
    # Save the current matplotlib figure as PNG + SVG into OUTPUT_DIR.
    plt.tight_layout()
    for ext in ("png", "svg"):
        plt.savefig(os.path.join(OUTPUT_DIR, f"{base_name}.{ext}"), dpi=dpi)
    plt.close()

# ── Load & clean ────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
df = df.drop(columns=["...7"], errors="ignore")
df["TF"] = df["TF"].astype(str).str.upper()
df["TF_Family"] = df["TF_Family"].astype(str)
df["Tissue"] = df["Tissue"].astype(str)

# ── Shared model frame with label-encoded categoricals ──────────────────────
df_model = df[["Gene", "TF", "Tissue", "log2FoldChange", "TF_Family"]].dropna().copy()
gene_le, tf_le, tissue_le = LabelEncoder(), LabelEncoder(), LabelEncoder()
df_model["Gene_encoded"]   = gene_le.fit_transform(df_model["Gene"].astype(str))
df_model["TF_encoded"]     = tf_le.fit_transform(df_model["TF"].astype(str))
df_model["Tissue_encoded"] = tissue_le.fit_transform(df_model["Tissue"].astype(str))
tf_family_map = df_model[["TF", "TF_Family"]].drop_duplicates()

df_model.head(50).to_csv(os.path.join(OUTPUT_DIR, "example_input_dataframe.csv"), index=False)
print(f"Loaded {len(df):,} rows; {len(df_model):,} complete rows used for modeling.")

## 1. Exploratory data analysis

In [ ]:
# Regulation class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=df["Regulation"])
plt.title("Regulation class distribution")
save_figure("regulation_class_distribution")

# log2FoldChange by TF family, split by regulation direction
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="TF_Family", y="log2FoldChange", hue="Regulation")
plt.xticks(rotation=90)
plt.title("Fold-change distribution by TF family")
save_figure("foldchange_by_tf_family")

# Per-tissue TF-family count distribution
for tissue in df["Tissue"].unique():
    tissue_df = df[df["Tissue"] == tissue]
    plt.figure(figsize=(10, 6))
    sns.countplot(y=tissue_df["TF_Family"], hue=tissue_df["Regulation"],
                  order=tissue_df["TF_Family"].value_counts().index)
    plt.title(f"TF-family distribution in {tissue}")
    save_figure(f"tf_family_distribution_{tissue}")

# Chi-square test: TF family vs regulation direction
contingency = pd.crosstab(df["TF_Family"], df["Regulation"])
chi2, p, dof, _ = chi2_contingency(contingency)
print(f"Chi-square: chi2={chi2:.4f}, p={p:.3e}, dof={dof}")

## 2. Exploratory classification of high-regulation genes

Label the top 30% of `log2FoldChange` as *high regulation* and predict it from fold-change, TF-family frequency, and tissue (Random Forest vs XGBoost).

In [ ]:
df_clf = df.dropna(subset=["log2FoldChange", "TF_Family", "Tissue"]).copy()
df_clf["TF_Family_Encoded"] = df_clf["TF_Family"].map(df_clf["TF_Family"].value_counts())
df_clf["Tissue_Encoded"] = LabelEncoder().fit_transform(df_clf["Tissue"])

threshold = np.percentile(df_clf["log2FoldChange"], 70)
df_clf["High_Regulation"] = (df_clf["log2FoldChange"] >= threshold).astype(int)

X = df_clf[["log2FoldChange", "TF_Family_Encoded", "Tissue_Encoded"]]
y = df_clf["High_Regulation"]
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

rf_clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42).fit(X_train, y_train)
print("Random Forest:\n", classification_report(y_test, rf_clf.predict(X_test)))

xgb_clf = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                            random_state=42, eval_metric="logloss").fit(X_train, y_train)
print("XGBoost:\n", classification_report(y_test, xgb_clf.predict(X_test)))

importance_df = pd.DataFrame({"Feature": X.columns, "Importance": xgb_clf.feature_importances_})
plt.figure(figsize=(6, 4))
sns.barplot(x="Importance", y="Feature", data=importance_df)
plt.title("Feature importance for high regulation (XGBoost)")
save_figure("xgboost_feature_importance")

## 3. TF-family up/down regulation bias (binomial odds-ratio)

Per tissue and TF family, test whether up- vs down-regulation deviates from 50/50 (binomial test, BH-adjusted), with log2 odds ratios and 95% CIs.

In [ ]:
counts = (df.groupby(["Tissue", "Regulation", "TF_Family"]).size()
            .reset_index(name="Count"))
pivot = counts.pivot_table(index=["Tissue", "TF_Family"], columns="Regulation",
                           values="Count", fill_value=0)

z = norm.ppf(0.975)
results = []
for (tissue, tf), row in pivot.iterrows():
    up, down = int(row.get("upregulated", 0)), int(row.get("downregulated", 0))
    odds_ratio = (up + 1) / (down + 1)          # pseudo-count for stability
    log2_or = np.log2(odds_ratio)
    se = np.sqrt(1 / (up + 1) + 1 / (down + 1))
    total = up + down
    pval = binomtest(up, total, p=0.5).pvalue if total > 0 else 1.0
    results.append({"Tissue": tissue, "TF_Family": tf, "Up": up, "Down": down,
                    "Total": total, "Odds_Up": odds_ratio, "log2_OR": log2_or,
                    "log2_OR_CI_L": log2_or - z * se, "log2_OR_CI_U": log2_or + z * se,
                    "p-value": pval})

results_df = pd.DataFrame(results)
results_df["p-adj"] = multipletests(results_df["p-value"], method="fdr_bh")[1]
results_df.to_csv(os.path.join(OUTPUT_DIR, "tf_family_binomial_results.csv"), index=False)

significant = results_df[results_df["p-adj"] < 0.05].copy()
if significant.empty:
    print("No TF families passed FDR < 0.05; CSV saved, skipping plots.")
else:
    tissues = list(significant["Tissue"].unique())

    # Plot 1: significant odds ratios per tissue
    fig, axes = plt.subplots(1, len(tissues), figsize=(7 * len(tissues), 6), squeeze=False)
    for ax, tissue in zip(axes[0], tissues):
        data = significant[significant["Tissue"] == tissue].sort_values("Odds_Up", ascending=False)
        sns.barplot(data=data, x="TF_Family", y="Odds_Up", ax=ax)
        ax.set_title(f"{tissue} – significant TF families")
        ax.set_ylabel("Odds ratio (up/down)")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        for c in ax.containers:
            ax.bar_label(c, labels=[f"{v:.2e}" for v in c.datavalues],
                         label_type="edge", fontsize=8, rotation=90, padding=3)
    save_figure("plot1_odds_ratio_bars")

    # Plot 2: log2(OR) with 95% CI
    tf_union = significant["TF_Family"].unique()
    plot_df = results_df[results_df["TF_Family"].isin(tf_union)].copy()
    order = plot_df.groupby("TF_Family")["log2_OR"].mean().sort_values().index
    plot_df["TF_Family"] = pd.Categorical(plot_df["TF_Family"], categories=order, ordered=True)
    plot_df = plot_df.sort_values(["TF_Family", "Tissue"])
    plt.figure(figsize=(14, 6))
    ax = sns.pointplot(data=plot_df, x="TF_Family", y="log2_OR", hue="Tissue",
                       dodge=0.4, join=False, errorbar=None)
    offsets = dict(zip(tissues, np.linspace(-0.2, 0.2, len(tissues))))
    for _, r in plot_df.iterrows():
        x = list(order).index(r["TF_Family"]) + offsets.get(r["Tissue"], 0.0)
        ax.plot([x, x], [r["log2_OR_CI_L"], r["log2_OR_CI_U"]], color="gray", linewidth=1)
    plt.axhline(0, color="black", linestyle="--")
    plt.xticks(rotation=45, ha="right")
    plt.title("TF-family log2 odds of up- vs down-regulation (95% CI)")
    plt.ylabel("log2(odds up/down)")
    plt.legend(title="Tissue", bbox_to_anchor=(1.01, 1), loc="upper left")
    save_figure("plot2_log2or_with_ci")

    # Plot 3: clustered heatmap of log2(OR)
    clust = results_df[results_df["TF_Family"].isin(tf_union)]
    hm = clust.pivot(index="TF_Family", columns="Tissue", values="log2_OR").fillna(0)
    mask = (clust.pivot(index="TF_Family", columns="Tissue", values="p-adj") >= 0.05
            ).reindex_like(hm).fillna(True)
    if len(hm) >= 2:                                  # clustering needs >= 2 rows
        link = linkage(pdist(hm.values), method="average")
        hm, mask = hm.iloc[leaves_list(link)], mask.iloc[leaves_list(link)]
        clusters = fcluster(link, min(3, len(hm)), criterion="maxclust")
        labels = [f"{tf} (C{c})" for tf, c in zip(hm.index, clusters)]
        hm.index = mask.index = labels
    plt.figure(figsize=(12, len(hm) * 0.5 + 2))
    sns.heatmap(hm, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                linewidths=0.6, linecolor="gray", mask=mask,
                cbar_kws={"label": "log2(odds up/down)"})
    plt.title("Clustered heatmap of TF-family regulation patterns")
    save_figure("plot3_clustered_heatmap")

## 4. LMC vs MEF TF-family expression differences (Welch t-test)

In [ ]:
df_t = df.dropna().copy()
lmc = df_t[df_t["Tissue"] == "LMC"].groupby("TF_Family")["log2FoldChange"].mean()
mef = df_t[df_t["Tissue"] == "MEF"].groupby("TF_Family")["log2FoldChange"].mean()
comp = (pd.concat([lmc, mef], axis=1, keys=["log2FoldChange_LMC", "log2FoldChange_MEF"])
          .dropna().reset_index())
comp["FoldChange_Diff"] = comp["log2FoldChange_LMC"] - comp["log2FoldChange_MEF"]

rows = []
for tf in df_t["TF_Family"].unique():
    a = df_t[(df_t.TF_Family == tf) & (df_t.Tissue == "LMC")]["log2FoldChange"]
    b = df_t[(df_t.TF_Family == tf) & (df_t.Tissue == "MEF")]["log2FoldChange"]
    if len(a) > 1 and len(b) > 1:
        t, p = ttest_ind(a, b, equal_var=False)
        rows.append({"TF_Family": tf, "t_stat": t, "p_value": p})
pv = pd.DataFrame(rows)
pv["Significant"] = pv["p_value"] < 0.05
comp = comp.merge(pv, on="TF_Family", how="left").sort_values("p_value")
comp.to_csv(os.path.join(OUTPUT_DIR, "tf_family_LMC_vs_MEF_statistical_analysis.csv"), index=False)
print(comp.head(20))

top = comp.head(15)
plt.figure(figsize=(12, 8))
plt.barh(top["TF_Family"], top["FoldChange_Diff"], color="#add8e6", edgecolor="black")
plt.gca().invert_yaxis()
plt.xlabel("Mean fold-change difference (LMC - MEF)")
plt.title("Top 15 TF families differing most between LMC and MEF")
for ypos, (_, r) in zip(range(len(top)), top.iterrows()):
    pstr = "p < 1e-100" if r["p_value"] < 1e-100 else f"p = {r['p_value']:.1e}"
    plt.text(r["FoldChange_Diff"], ypos, "  " + pstr, va="center", fontsize=9)
save_figure("tf_family_LMC_vs_MEF_top15")

## 5. Random-forest regression of log2FoldChange — Supplementary Figure 2c

Over 10 random subsamples (n=20,000 each), train a Random Forest to predict `log2FoldChange` from gene, TF, and tissue. Records 5-fold CV and held-out test R²/MSE distributions and consensus feature importance.

In [ ]:
NUM_ITERATIONS = 10
N_ESTIMATORS = 500
SAMPLE_N = min(20000, len(df_model))
FEATURES = ["Gene_encoded", "TF_encoded", "Tissue_encoded"]

cv_r2, cv_mse, test_r2, test_mse, importances = [], [], [], [], []
final_predictions_df = final_training_df = None

for i in range(NUM_ITERATIONS):
    sample = df_model.sample(n=SAMPLE_N, random_state=i)
    X, y = sample[FEATURES], sample["log2FoldChange"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=i)

    rf = RandomForestRegressor(n_estimators=N_ESTIMATORS, n_jobs=-1, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)
    cv_r2.append(cross_val_score(rf, X_tr, y_tr, cv=kf, scoring="r2").mean())
    cv_mse.append(-cross_val_score(rf, X_tr, y_tr, cv=kf, scoring="neg_mean_squared_error").mean())

    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)
    test_r2.append(r2_score(y_te, y_pred))
    test_mse.append(mean_squared_error(y_te, y_pred))
    importances.append(rf.feature_importances_)

    if i == NUM_ITERATIONS - 1:                       # keep detailed outputs from last run
        te = X_te.copy()
        te["Actual_log2FoldChange"] = y_te.values
        te["Predicted_log2FoldChange"] = y_pred
        te["Gene"]   = gene_le.inverse_transform(te["Gene_encoded"])
        te["TF"]     = tf_le.inverse_transform(te["TF_encoded"])
        te["Tissue"] = tissue_le.inverse_transform(te["Tissue_encoded"])
        final_predictions_df = te.merge(tf_family_map, on="TF", how="left")

        tr = X_tr.copy()
        tr["log2FoldChange"] = y_tr.values
        tr["Gene"]   = gene_le.inverse_transform(tr["Gene_encoded"])
        tr["TF"]     = tf_le.inverse_transform(tr["TF_encoded"])
        tr["Tissue"] = tissue_le.inverse_transform(tr["Tissue_encoded"])
        final_training_df = tr.merge(tf_family_map, on="TF", how="left")

print("Summary over iterations")
print(f"CV   R²:  {np.mean(cv_r2):.4f} +/- {np.std(cv_r2):.4f}")
print(f"Test R²:  {np.mean(test_r2):.4f} +/- {np.std(test_r2):.4f}")
print(f"Test MSE: {np.mean(test_mse):.4f} +/- {np.std(test_mse):.4f}")

final_predictions_df.to_csv(os.path.join(OUTPUT_DIR, "random_forest_predictions_with_tf_family.csv"), index=False)
final_training_df.to_csv(os.path.join(OUTPUT_DIR, "random_forest_training_set.csv"), index=False)

metrics_df = pd.DataFrame({"Iteration": range(1, NUM_ITERATIONS + 1),
                           "CV_R2": cv_r2, "CV_MSE": cv_mse,
                           "Test_R2": test_r2, "Test_MSE": test_mse})
summary_row = pd.DataFrame([{"Iteration": "Mean +/- Std",
    "CV_R2": f"{np.mean(cv_r2):.4f} +/- {np.std(cv_r2):.4f}",
    "CV_MSE": f"{np.mean(cv_mse):.4f} +/- {np.std(cv_mse):.4f}",
    "Test_R2": f"{np.mean(test_r2):.4f} +/- {np.std(test_r2):.4f}",
    "Test_MSE": f"{np.mean(test_mse):.4f} +/- {np.std(test_mse):.4f}"}])
pd.concat([metrics_df, summary_row], ignore_index=True).to_csv(
    os.path.join(OUTPUT_DIR, "iteration_metrics_summary.csv"), index=False)

In [ ]:
# Supplementary Figure 2c: R²/MSE distributions + consensus feature importance
sns.set_theme(style="white")
fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor="white")
panels = [(axes[0, 0], cv_r2,   "R²",  "steelblue", "Cross-Validation R²"),
          (axes[0, 1], test_r2, "R²",  "steelblue", "Test R²"),
          (axes[1, 0], cv_mse,  "MSE", "salmon",    "Cross-Validation MSE"),
          (axes[1, 1], test_mse,"MSE", "salmon",    "Test MSE")]
for ax, scores, ylabel, color, title in panels:
    d = pd.DataFrame({ylabel: scores})
    sns.boxplot(data=d, y=ylabel, ax=ax, color=color, width=0.45, boxprops=dict(alpha=0.7))
    sns.stripplot(data=d, y=ylabel, ax=ax, color="black", size=7, jitter=True, alpha=0.8)
    ax.set_title(title, fontweight="bold")
    sns.despine(ax=ax, trim=True)

ax_fi = axes[0, 2]
fi_mean = np.mean(importances, axis=0)
fi_std = np.std(importances, axis=0)
bars = ax_fi.barh(["Gene", "TF Motif", "Cell Type"], fi_mean, xerr=fi_std,
                  color=["#4C72B0", "#55A868", "#C44E52"], alpha=0.8,
                  capsize=4, error_kw=dict(elinewidth=1.2, ecolor="black"))
for bar, val in zip(bars, fi_mean):
    ax_fi.text(val + 0.005, bar.get_y() + bar.get_height() / 2, f"{val*100:.1f}%",
               va="center", fontweight="bold")
ax_fi.set_title("Feature importance\n(Random Forest)", fontweight="bold")
ax_fi.set_xlabel("Mean importance score")
sns.despine(ax=ax_fi, trim=True)
axes[1, 2].set_visible(False)
plt.suptitle("Random Forest performance across 10 iterations", fontweight="bold", y=1.01)
save_figure("boxplots_r2_mse")
sns.set_theme(style="whitegrid")

## 6. Robustness: ablation & permutation importance of TF

Across 10 splits, measure the TF feature's contribution two ways: permutation ΔR² and the R² drop when TF is removed from the model.

In [ ]:
perm_deltas, abl_deltas = [], []
for seed in range(10):
    sample = df_model.sample(n=min(20000, len(df_model)), random_state=seed)
    X = sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]]
    y = sample["log2FoldChange"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)

    rf_full = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=seed).fit(X_tr, y_tr)
    r2_full = r2_score(y_te, rf_full.predict(X_te))

    perm = permutation_importance(rf_full, X_te, y_te, scoring="r2",
                                  n_repeats=30, random_state=seed, n_jobs=-1)
    perm_deltas.append(perm.importances_mean[list(X_te.columns).index("TF_encoded")])

    rf_noTF = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=seed)
    rf_noTF.fit(X_tr.drop(columns="TF_encoded"), y_tr)
    abl_deltas.append(r2_full - r2_score(y_te, rf_noTF.predict(X_te.drop(columns="TF_encoded"))))

print(f"Permutation ΔR² for TF : {np.mean(perm_deltas):.4f} +/- {np.std(perm_deltas):.4f}")
print(f"Ablation ΔR² (full-noTF): {np.mean(abl_deltas):.4f} +/- {np.std(abl_deltas):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=perm_deltas, ax=ax[0]); ax[0].set_title("Permutation ΔR² (TF)")
sns.boxplot(y=abl_deltas, ax=ax[1]); ax[1].set_title("Ablation ΔR² (full - noTF)")
save_figure("tf_ablation_permutation")

## 7. Feature-set comparison

Compare test R² for increasingly informative feature sets, including a composite Gene×TF identifier.

In [ ]:
fc = df_model.copy()
fc["Gene_TF_enc"] = LabelEncoder().fit_transform(fc["Gene"] + "_" + fc["TF"])

feature_sets = {
    "gene_only":        ["Gene_encoded"],
    "gene_plus_TF":     ["Gene_encoded", "TF_encoded"],
    "composite_only":   ["Gene_TF_enc"],
    "composite+tissue": ["Gene_TF_enc", "Tissue_encoded"],
    "full":             ["Gene_encoded", "TF_encoded", "Tissue_encoded"],
}
results = {name: [] for name in feature_sets}
for seed in range(10):
    data = fc.sample(n=min(20000, len(fc)), random_state=seed)
    y = data["log2FoldChange"]
    tr_idx, te_idx = train_test_split(data.index, test_size=0.2, random_state=seed)
    for name, feats in feature_sets.items():
        Xtr = data.loc[tr_idx, feats].astype("category")
        Xte = data.loc[te_idx, feats].astype("category")
        rf = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=seed)
        rf.fit(Xtr, y.loc[tr_idx])
        results[name].append(r2_score(y.loc[te_idx], rf.predict(Xte)))

summary = (pd.DataFrame(results).agg(["mean", "std"]).T
             .rename(columns={"mean": "R2_mean", "std": "R2_std"}))
print(summary)

rename = {"gene_only": "Gene only", "gene_plus_TF": "Gene + TF",
          "composite_only": "Composite only", "composite+tissue": "Composite + Tissue",
          "full": "Full"}
df_plot = pd.DataFrame(results).rename(columns=rename)
order = [rename[o] for o in summary.sort_values("R2_mean").index]
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_plot, order=order, palette="pastel")
sns.stripplot(data=df_plot, order=order, color="gray", size=3, jitter=True)
for i, col in enumerate(order):
    m = df_plot[col].median()
    plt.text(i, m, f"{m:.2f}", ha="center", va="center", fontweight="bold",
             bbox=dict(facecolor="white", alpha=0.7, boxstyle="round,pad=0.2"))
plt.ylabel("Test $R^2$")
plt.title("Feature-set comparison: test $R^2$ distribution")
plt.xticks(rotation=45)
save_figure("feature_set_comparison_r2")

## 8. Leakage-safe target-encoded model

Replace TF and TF-family with out-of-fold target encodings (means computed only from training folds) to avoid leakage, then re-evaluate.

In [ ]:
def oof_target_encode(train_df, test_df, cat_col, y_col, n_splits=5, seed=0):
    # Out-of-fold mean target encoding; unseen categories fall back to the global mean.
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = pd.Series(np.nan, index=train_df.index, dtype=np.float32)
    global_mean = train_df[y_col].mean()
    for tr_idx, val_idx in kf.split(train_df):
        means = train_df.iloc[tr_idx].groupby(cat_col)[y_col].mean()
        oof.iloc[val_idx] = train_df.iloc[val_idx][cat_col].map(means).fillna(global_mean).values
    full_means = train_df.groupby(cat_col)[y_col].mean()
    test_te = test_df[cat_col].map(full_means).fillna(global_mean).astype(np.float32)
    return oof.astype(np.float32), test_te

cv_r2s, te_r2s, perm_tf, abl = [], [], [], []
final_pred = final_train = None
for seed in range(10):
    sub = df_model.sample(n=min(20000, len(df_model)), random_state=seed).reset_index(drop=True)
    y = sub["log2FoldChange"].astype(np.float32)
    tr_idx, te_idx = train_test_split(sub.index, test_size=0.2, random_state=seed)

    enc = sub[["TF", "TF_Family", "log2FoldChange"]].rename(columns={"log2FoldChange": "y"})
    tf_tr,  tf_te  = oof_target_encode(enc.loc[tr_idx], enc.loc[te_idx], "TF", "y", seed=seed)
    fam_tr, fam_te = oof_target_encode(enc.loc[tr_idx], enc.loc[te_idx], "TF_Family", "y", seed=seed)

    Xtr = pd.DataFrame({"Gene_enc": sub.loc[tr_idx, "Gene_encoded"].values,
                        "Tissue_enc": sub.loc[tr_idx, "Tissue_encoded"].values,
                        "TF_te": tf_tr.values, "TF_Family_te": fam_tr.values})
    Xte = pd.DataFrame({"Gene_enc": sub.loc[te_idx, "Gene_encoded"].values,
                        "Tissue_enc": sub.loc[te_idx, "Tissue_encoded"].values,
                        "TF_te": tf_te.values, "TF_Family_te": fam_te.values})
    ytr, yte = y.loc[tr_idx].values, y.loc[te_idx].values

    rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=seed)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    cv_r2s.append(cross_val_score(rf, Xtr, ytr, cv=kf, scoring="r2").mean())
    rf.fit(Xtr, ytr)
    yp = rf.predict(Xte)
    te_r2s.append(r2_score(yte, yp))

    perm = permutation_importance(rf, Xte, yte, scoring="r2", n_repeats=10, random_state=seed, n_jobs=-1)
    perm_tf.append(perm.importances_mean[list(Xte.columns).index("TF_te")])
    rf_noTF = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=seed)
    rf_noTF.fit(Xtr[["Gene_enc", "Tissue_enc"]], ytr)
    abl.append(r2_score(yte, yp) - r2_score(yte, rf_noTF.predict(Xte[["Gene_enc", "Tissue_enc"]])))

    if seed == 9:
        final_pred = Xte.assign(Actual_log2FoldChange=yte, Predicted_log2FoldChange=yp,
                                Gene=sub.loc[te_idx, "Gene"].values, TF=sub.loc[te_idx, "TF"].values,
                                TF_Family=sub.loc[te_idx, "TF_Family"].values)
        final_train = Xtr.assign(log2FoldChange=ytr, Gene=sub.loc[tr_idx, "Gene"].values,
                                 TF=sub.loc[tr_idx, "TF"].values,
                                 TF_Family=sub.loc[tr_idx, "TF_Family"].values)

print(f"CV R²  : {np.mean(cv_r2s):.4f} +/- {np.std(cv_r2s):.4f}")
print(f"Test R²: {np.mean(te_r2s):.4f} +/- {np.std(te_r2s):.4f}")
print(f"Perm ΔR² (TF_te)               : {np.mean(perm_tf):.4f} +/- {np.std(perm_tf):.4f}")
print(f"Ablation ΔR² (full - no TF/Fam): {np.mean(abl):.4f} +/- {np.std(abl):.4f}")
final_pred.to_csv(os.path.join(OUTPUT_DIR, "rf_predictions_leakage_safe_te.csv"), index=False)
final_train.to_csv(os.path.join(OUTPUT_DIR, "rf_training_leakage_safe_te.csv"), index=False)

## 9. Generalization to unseen genes (gene-level holdout)

Hold out 20% of genes entirely (no gene in both train and test) to test generalization rather than memorization.

In [ ]:
rng = np.random.RandomState(42)
genes = df_model["Gene_encoded"].unique()
test_genes = set(rng.choice(genes, size=int(0.2 * len(genes)), replace=False))
is_test = df_model["Gene_encoded"].isin(test_genes)

feats = ["Gene_encoded", "TF_encoded", "Tissue_encoded"]
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(df_model.loc[~is_test, feats], df_model.loc[~is_test, "log2FoldChange"])
y_pred = rf.predict(df_model.loc[is_test, feats])
y_true = df_model.loc[is_test, "log2FoldChange"]

print("Performance on genes not seen in training:")
print(f"R²:  {r2_score(y_true, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_true, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.4f}")

## 10. Mixed-effects model

Random intercept + random tissue slope per gene–TF pair, modeling `log2FoldChange` as a function of tissue.

In [ ]:
mm = df.dropna(subset=["Gene", "TF", "Tissue", "log2FoldChange"]).copy()
mm["composite"] = mm["Gene"] + "_" + mm["TF"]
model = smf.mixedlm("log2FoldChange ~ C(Tissue)", mm, groups=mm["composite"],
                    re_formula="~C(Tissue)")
print(model.fit(method="lbfgs").summary())